In [1]:
from pathlib import Path
import pandas as pd

def load_rmats_txts(root_dir, sep="\t", **read_csv_kwargs):
    """
    root_dir/
      ├── output_N_vs_A/
      │     ├── A3SS.MATS.JCEC.txt
      │     ├── A5SS.MATS.JCEC.txt
      │     ├── MXE.MATS.JCEC.txt
      │     ├── RI.MATS.JCEC.txt
      │     ├── SE.MATS.JCEC.txt
      │     └── summary.txt
      ├── output_X_vs_Y/
      └── ...

    Creates dfs like:
      N_vs_A_A3SS, N_vs_A_A5SS, ...
    """

    root_path = Path(root_dir)
    dfs = {}

    for subdir in root_path.iterdir():
        if not subdir.is_dir():
            continue

        orig_dir_name = subdir.name          # e.g. 'output_N_vs_A'
        dir_name = orig_dir_name
        if dir_name.startswith("output_"):
            dir_name = dir_name[len("output_"):]   # -> 'N_vs_A'

        for txt_file in subdir.glob("*.txt"):
            # e.g. 'A3SS.MATS.JCEC.txt' -> 'A3SS'
            fileprefix = txt_file.name.split(".")[0]

            df_name = f"{dir_name}_{fileprefix}"   # e.g. 'N_vs_A_A3SS'

            df = pd.read_csv(txt_file, sep=sep, **read_csv_kwargs)
            dfs[df_name] = df

    return dfs

# Example usage:
root_dir = "/home/maggy/PhD/WholeGenomeBisulphiteSequencing/RNAseq_pipeline/17_splicing/01_rMATS"
dfs = load_rmats_txts(root_dir, sep="\t")

# Example: directory 'output_N_vs_A' + file 'A3SS.MATS.JCEC.txt'
df = dfs["N_vs_A_A3SS"]
print(df.head())


   ID              GeneID  geneSymbol             chr strand  \
0   0  scaffold_49-g45725         NaN  chrscaffold_49      +   
1   1  scaffold_44-g45718         NaN  chrscaffold_44      -   
2   2  scaffold_30-g45662         NaN  chrscaffold_30      -   
3   3    ccar_ub25-g45508         NaN    chrccar_ub25      -   
4   4    ccar_ub25-g45499         NaN    chrccar_ub25      +   

   longExonStart_0base  longExonEnd   shortES   shortEE  flankingES  ...  \
0               111029       111142    111050    111142      107079  ...   
1                90722        90801     90722     90798       90885  ...   
2                67761        68091     67761     68076       71320  ...   
3             26467759     26467832  26467759  26467829    26467923  ...   
4             26002058     26002140  26002088  26002140    26000612  ...   

   SJC_SAMPLE_1     IJC_SAMPLE_2 SJC_SAMPLE_2 IncFormLen SkipFormLen  \
0       0,0,0,1          6,7,5,5      0,0,0,0        170         149   
1       0,5,3,

In [2]:
dfs.keys()

dict_keys(['24R_vs_A_SE', '24R_vs_A_MXE', '24R_vs_A_RI', '24R_vs_A_A5SS', '24R_vs_A_A3SS', '24R_vs_A_summary', 'N_vs_24R_SE', 'N_vs_24R_MXE', 'N_vs_24R_RI', 'N_vs_24R_A5SS', 'N_vs_24R_A3SS', 'N_vs_24R_summary', 'N_vs_A_SE', 'N_vs_A_MXE', 'N_vs_A_RI', 'N_vs_A_A5SS', 'N_vs_A_A3SS', 'N_vs_A_summary'])

In [ ]:
# dfs is your original dict
# Make a new dict with "24R" -> "R" in the keys
dfs_renamed = {
    key.replace("24R", "R"): value
    for key, value in dfs.items()
}

# If you want to replace the original dict:
dfs = dfs_renamed


In [ ]:
# dfs is your dict: {'N_vs_A_A3SS': df1, 'N_vs_A_A5SS': df2, ...}
for name, df in dfs.items():
    globals()[name] = df


In [18]:
# import dmr genes saves as set in a list format 
dmr_genes_set = pd.read_csv('/home/maggy/PhD/WholeGenomeBisulphiteSequencing/RNAseq_pipeline/17_splicing/01_rMATS/17_01_01_dmr_listing.txt', header=None)
dmr_genes_set = set(dmr_genes_set[0].astype(str).str.strip().to_list())



In [19]:
len(dmr_genes_set)

53

In [20]:
dfs_dmr = {}  # new dict for filtered DataFrames

for name, df in dfs.items():
    # keep only rows whose GeneID is in dmr_genes_set
    if "GeneID" not in df.columns:
        # skip if this df doesn't have GeneID (e.g. summary)
        continue

    df_filtered = df.loc[df["GeneID"].isin(dmr_genes_set)]

    # Optionally skip empty results
    if df_filtered.empty:
        continue

    dfs_dmr[name] = df_filtered

# Now dfs_dmr has the same keys as dfs (for those with matches),
# but each DataFrame contains only rows with GeneID in dmr_genes.

In [21]:
df_filtered

,ID,GeneID,geneSymbol,chr,strand,longExonStart_0base,longExonEnd,shortES,shortEE,flankingES,...,SJC_SAMPLE_1,IJC_SAMPLE_2,SJC_SAMPLE_2,IncFormLen,SkipFormLen,PValue,FDR,IncLevel1,IncLevel2,IncLevelDifference
2738,3146,ccar_ub04-g26931,NaN,chrccar_ub04,+,1612488,1612558,1612491,1612558,1610674,...,"19,5,0,6","0,0,0,0","2,6,2,1",152,149,0.454238,0.977881,"0.0,0.164,1.0,0.0","0.0,0.0,0.0,0.0",0.291
2977,3416,ccar_ua15-g14343,NaN,chrccar_ua15,+,4482979,4483214,4482994,4483214,4461895,...,"12,8,3,4","3,2,12,8","6,5,0,1",164,149,0.804211,1.000000,"0.532,0.723,0.887,0.82","0.312,0.267,1.0,0.879",0.126
3038,3480,ccar_ub12-g34593,NaN,chrccar_ub12,-,11307567,11307739,11307567,11307655,11310759,...,"9,5,3,3","28,10,14,19","2,6,1,8",233,149,0.789481,1.000000,"0.63,0.729,0.882,0.824","0.9,0.516,0.9,0.603",0.037
4293,4904,ccar_ua19-g18077,NaN,chrccar_ua19,+,18867491,18867595,18867494,18867595,18865676,...,"87,52,51,43","152,72,78,136","62,40,56,38",152,149,0.046280,0.256897,"0.75,0.76,0.765,0.726","0.706,0.638,0.577,0.778",0.076
4294,4905,ccar_ua19-g18077,NaN,chrccar_ua19,+,18869548,18869655,18869572,18869655,18867491,...,"112,71,85,62","245,114,116,215","51,29,35,66",173,149,0.018433,0.142770,"0.718,0.743,0.653,0.715","0.805,0.772,0.741,0.737",-0.057


In [22]:
dfs_dmr.keys()

dict_keys(['R_vs_A_SE', 'R_vs_A_MXE', 'R_vs_A_RI', 'R_vs_A_A5SS', 'R_vs_A_A3SS', 'N_vs_R_SE', 'N_vs_R_MXE', 'N_vs_R_RI', 'N_vs_R_A5SS', 'N_vs_R_A3SS', 'N_vs_A_SE', 'N_vs_A_MXE', 'N_vs_A_RI', 'N_vs_A_A5SS', 'N_vs_A_A3SS'])

In [33]:
def filter_significant(dfs_in, p_col="PValue", fdr_col="FDR",
                       p_thr=0.05, fdr_thr=0.05):
    dfs_sig = {}
    for name, df in dfs_in.items():
        # skip dfs that lack the required columns
        if p_col not in df.columns or fdr_col not in df.columns:
            continue

        df_sig = df.loc[(df[p_col] < p_thr) & (df[fdr_col] < fdr_thr)]

        # optionally drop empty ones
        if df_sig.empty:
            continue

        dfs_sig[name] = df_sig

    return dfs_sig

# significant events in all rMATS outputs
dfs_sig = filter_significant(dfs)

# significant events among DMR-overlapping genes
dfs_dmr_sig = filter_significant(dfs_dmr)


In [34]:
for name, df in dfs_dmr_sig.items():
    print(name, len(df))


R_vs_A_RI 4
N_vs_R_SE 2
N_vs_R_MXE 1
N_vs_A_SE 4
N_vs_A_RI 3
N_vs_A_A5SS 1


In [32]:
R_vs_A_SE.loc[(R_vs_A_SE['PValue'] < 0.05) & (R_vs_A_SE['FDR'] < 0.05)]


,ID,GeneID,geneSymbol,chr,strand,exonStart_0base,exonEnd,upstreamES,upstreamEE,downstreamES,...,SJC_SAMPLE_1,IJC_SAMPLE_2,SJC_SAMPLE_2,IncFormLen,SkipFormLen,PValue,FDR,IncLevel1,IncLevel2,IncLevelDifference
1,1,scaffold_59-g45738,NaN,chrscaffold_59,+,10143,10179,8493,8592,13041,...,"45,35,68,42","21,3,15,12","40,33,30,34",185,149,9.666181e-04,0.048170,"0.067,0.084,0.115,0.118","0.297,0.068,0.287,0.221",-0.122
78,83,ccar_ub25-g45438,NaN,chrccar_ub25,+,22680037,22680117,22679788,22679914,22690658,...,"4,1,1,4","3,9,12,8","0,0,1,0",229,149,1.593809e-05,0.002761,"0.839,0.901,0.877,0.532","1.0,1.0,0.886,1.0",-0.184
131,136,ccar_ub25-g45353,NaN,chrccar_ub25,-,19603599,19603751,19601742,19602023,19609633,...,"9,5,10,5","96,90,114,91","1,0,4,2",301,149,6.103445e-05,0.006981,"0.881,0.9,0.84,0.913","0.979,1.0,0.934,0.957",-0.084
132,137,ccar_ub25-g45353,NaN,chrccar_ub25,-,19606422,19606577,19601742,19602023,19609633,...,"9,5,10,5","94,75,97,83","1,0,4,2",304,149,1.041033e-04,0.010595,"0.906,0.901,0.769,0.866","0.979,1.0,0.922,0.953",-0.103
180,188,ccar_ub25-g45242,NaN,chrccar_ub25,-,16040069,16040123,16036258,16036377,16040726,...,"66,25,25,8","484,237,309,372","23,7,7,12",203,149,3.031563e-04,0.022154,"0.847,0.933,0.927,0.971","0.939,0.961,0.97,0.958",-0.038
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
35552,36810,ccar_ua22-g20911,NaN,chrccar_ua22,-,26924823,26925125,26924585,26924717,26925446,...,"23,7,6,6","408,925,513,689","56,11,88,14",451,149,5.841346e-10,0.000002,"0.956,0.98,0.973,0.982","0.706,0.965,0.658,0.942",0.155
35649,36910,ccar_ub23-g43558,NaN,chrccar_ub23,+,11029566,11029639,11026603,11026690,11031536,...,"7,11,2,2","2,4,2,2","10,9,8,6",222,149,6.649985e-04,0.037318,"0.0,0.058,0.251,0.0","0.118,0.23,0.144,0.183",-0.092
35869,37137,ccar_ub16-g38174,NaN,chrccar_ub16,+,14067637,14067703,14055106,14055305,14067890,...,"9,26,21,14","63,41,60,79","2,3,8,0",215,149,5.380793e-08,0.000059,"0.806,0.607,0.748,0.745","0.956,0.905,0.839,1.0",-0.199
35883,37152,ccar_ub23-g44036,NaN,chrccar_ub23,-,26282020,26282096,26281625,26281684,26286457,...,"0,2,0,3","36,17,18,38","6,0,5,6",225,149,6.090883e-04,0.035393,"1.0,0.874,1.0,0.872","0.799,1.0,0.704,0.807",0.109
